## Example 1: Basic JSON logging setup and configuration

In [ ]:
from pythonjsonlogger.json import JsonFormatter
import logging
import io
import json

# Example 1: Basic JSON logging setup and configuration
print("Testing python-json-logger basic JSON logging:")

try:
    # JSON logger components
    print(f"✓ JsonFormatter: {JsonFormatter.__module__}.{JsonFormatter.__qualname__}")
    assert issubclass(JsonFormatter, logging.Formatter)
    
    # Create JSON formatter
    json_formatter = JsonFormatter()
    print(f"✓ JSON formatter created: {type(json_formatter)}")
    
    # Set up logger with JSON output
    logger = logging.getLogger('example_json_logger')
    logger.setLevel(logging.DEBUG)
    
    # Create string stream to capture output for demonstration
    log_stream = io.StringIO()
    handler = logging.StreamHandler(log_stream)
    handler.setFormatter(json_formatter)
    
    # Clear any existing handlers and add our JSON handler
    logger.handlers.clear()
    logger.addHandler(handler)
    
    print(f"✓ Logger configured with JSON formatter")
    
    # Test different log levels
    log_messages = [
        ("DEBUG", "Debug message for troubleshooting"),
        ("INFO", "Information message about application state"),
        ("WARNING", "Warning about potential issue"),
        ("ERROR", "Error occurred in processing"),
        ("CRITICAL", "Critical system failure")
    ]
    
    print(f"✓ Testing different log levels:")
    
    for level, message in log_messages:
        # Log with the appropriate level
        if level == "DEBUG":
            logger.debug(message)
        elif level == "INFO":
            logger.info(message)
        elif level == "WARNING":
            logger.warning(message)
        elif level == "ERROR":
            logger.error(message)
        elif level == "CRITICAL":
            logger.critical(message)
    
    # Get logged output and parse JSON
    log_output = log_stream.getvalue()
    log_lines = [line.strip() for line in log_output.split('\n') if line.strip()]
    
    print(f"    Generated {len(log_lines)} JSON log entries")
    
    # Parse and display first few JSON log entries
    for i, line in enumerate(log_lines[:3]):
        try:
            log_entry = json.loads(line)
            print(f"    Entry {i+1}: {log_entry.get('levelname', 'N/A')} - {log_entry.get('message', 'N/A')[:50]}...")
        except json.JSONDecodeError:
            print(f"    Entry {i+1}: Invalid JSON - {line[:50]}...")
    
    # Verify JSON structure
    if log_lines:
        sample_entry = json.loads(log_lines[0])
        print(f"✓ JSON log entry structure:")
        for key in sorted(sample_entry.keys()):
            print(f"    {key}: {type(sample_entry[key]).__name__}")
    
    # Clean up
    logger.removeHandler(handler)
    handler.close()
    log_stream.close()
    
    print("Example 1 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e).__name__}")
    raise AssertionError(
        f"python_json_logger example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Custom JSON formatter with extra fields and structured data

In [ ]:
from pythonjsonlogger.json import JsonFormatter
import logging
import io
import json
import datetime

# Example 2: Custom JSON formatter with extra fields and structured data
print("Testing python-json-logger with structured data and custom fields:")

try:
    # Create custom JSON formatter with specific fields
    custom_formatter = JsonFormatter(
        fmt='%(asctime)s %(name)s %(levelname)s %(message)s'
    )
    
    print(f"✓ Custom JSON formatter created with timestamp and name fields")
    
    # Set up logger for structured logging
    struct_logger = logging.getLogger('structured_app_logger')
    struct_logger.setLevel(logging.INFO)
    
    # Create handler with custom formatter
    log_stream = io.StringIO()
    handler = logging.StreamHandler(log_stream)
    handler.setFormatter(custom_formatter)
    
    # Configure logger
    struct_logger.handlers.clear()
    struct_logger.addHandler(handler)
    
    print(f"✓ Structured logger configured")
    
    # Log with extra structured data
    structured_logs = [
        {
            'message': 'User authentication successful',
            'extra': {
                'user_id': 12345,
                'username': 'john_doe',
                'ip_address': '192.168.1.100',
                'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
                'login_method': 'password'
            }
        },
        {
            'message': 'Database query executed',
            'extra': {
                'query_type': 'SELECT',
                'table': 'users',
                'duration_ms': 45.2,
                'rows_returned': 1,
                'query_id': 'q_789'
            }
        },
        {
            'message': 'API request processed',
            'extra': {
                'endpoint': '/api/v1/users',
                'method': 'GET',
                'response_code': 200,
                'response_time_ms': 123.5,
                'request_id': 'req_456'
            }
        }
    ]
    
    print(f"✓ Logging structured data:")
    
    for i, log_data in enumerate(structured_logs):
        struct_logger.info(log_data['message'], extra=log_data['extra'])
        print(f"    Entry {i+1}: {log_data['message'][:40]}...")
    
    # Parse and analyze structured logs
    log_output = log_stream.getvalue()
    log_lines = [line.strip() for line in log_output.split('\n') if line.strip()]
    
    print(f"✓ Generated {len(log_lines)} structured JSON log entries")
    
    # Analyze structured data
    for i, line in enumerate(log_lines):
        try:
            log_entry = json.loads(line)
            print(f"    Entry {i+1} analysis:")
            print(f"      Level: {log_entry.get('levelname')}")
            print(f"      Message: {log_entry.get('message')}")
            print(f"      Timestamp: {log_entry.get('asctime', 'N/A')}")
            
            # Count extra fields (excluding standard logging fields)
            standard_fields = {'asctime', 'name', 'levelname', 'message'}
            extra_fields = set(log_entry.keys()) - standard_fields
            print(f"      Extra fields: {len(extra_fields)} ({', '.join(sorted(extra_fields))})")
            
            # Show sample extra data
            if extra_fields:
                sample_field = next(iter(extra_fields))
                print(f"      Sample data: {sample_field} = {log_entry.get(sample_field)}")
                
        except json.JSONDecodeError as e:
            print(f"    Entry {i+1}: JSON parse error - {e}")
    
    # Demonstrate log filtering and searching capabilities
    print(f"✓ Structured data analysis capabilities:")
    
    parsed_logs = []
    for line in log_lines:
        try:
            parsed_logs.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    
    # Filter by user_id
    user_logs = [log for log in parsed_logs if 'user_id' in log]
    print(f"    User-related logs: {len(user_logs)}")
    
    # Filter by response time
    performance_logs = [log for log in parsed_logs if 'response_time_ms' in log]
    print(f"    Performance logs: {len(performance_logs)}")
    
    # Calculate metrics from structured data
    response_times = [log.get('response_time_ms') for log in performance_logs if log.get('response_time_ms')]
    if response_times:
        avg_response = sum(response_times) / len(response_times)
        print(f"    Average response time: {avg_response:.2f}ms")
    
    # Clean up
    struct_logger.removeHandler(handler)
    handler.close()
    log_stream.close()
    
    print("Example 2 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e).__name__}")
    raise AssertionError(
        f"python_json_logger example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: JSON formatter configurations and custom serialization

In [ ]:
from pythonjsonlogger.json import JsonFormatter
import logging
import io
import json

# Example 3: JSON formatter configurations and custom serialization
print("Testing python-json-logger advanced configurations:")

try:
    # Test different JSON formatter configurations
    formatter_configs = [
        {
            'name': 'minimal',
            'formatter': JsonFormatter(fmt='%(message)s'),
            'description': 'Minimal formatter with message only'
        },
        {
            'name': 'standard',
            'formatter': JsonFormatter(fmt='%(asctime)s %(levelname)s %(message)s'),
            'description': 'Standard formatter with timestamp and level'
        },
        {
            'name': 'detailed',
            'formatter': JsonFormatter(
                fmt='%(asctime)s %(name)s %(levelname)s %(funcName)s %(lineno)d %(message)s'
            ),
            'description': 'Detailed formatter with function and line info'
        },
        {
            'name': 'custom_prefix',
            'formatter': JsonFormatter(
                fmt='%(asctime)s %(levelname)s %(message)s',
                prefix='app'
            ),
            'description': 'Formatter with custom prefix'
        }
    ]
    
    print(f"✓ Testing {len(formatter_configs)} formatter configurations:")
    
    # Test each formatter configuration
    test_results = {}
    
    for config in formatter_configs:
        logger = logging.getLogger(f'test_{config["name"]}')
        logger.setLevel(logging.INFO)
        
        # Create stream and handler for this config
        log_stream = io.StringIO()
        handler = logging.StreamHandler(log_stream)
        handler.setFormatter(config['formatter'])
        
        logger.handlers.clear()
        logger.addHandler(handler)
        
        # Test log message
        logger.info('Test message for configuration', extra={'config_name': config['name']})
        
        # Get output
        output = log_stream.getvalue().strip()
        # prefix= is written verbatim before the JSON object, so strip it before parsing.
        prefix = config['formatter'].prefix
        assert output.startswith(prefix), output
        
        try:
            parsed_output = json.loads(output[len(prefix):])
            test_results[config['name']] = {
                'success': True,
                'fields': list(parsed_output.keys()),
                'sample': parsed_output
            }
            print(f"    {config['name']}: ✓ {len(parsed_output.keys())} fields")
            print(f"      Fields: {', '.join(parsed_output.keys())}")
            assert parsed_output['message'] == 'Test message for configuration', parsed_output
            assert parsed_output['config_name'] == config['name'], parsed_output
            
        except json.JSONDecodeError as e:
            raise AssertionError(f"{config['name']} formatter emitted a line that is not JSON: {output!r}") from e
        
        # Clean up
        logger.removeHandler(handler)
        handler.close()
        log_stream.close()
    
    # Test custom JSON encoder
    print(f"✓ Testing custom JSON serialization:")
    
    class CustomEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, datetime.datetime):
                return obj.isoformat()
            elif isinstance(obj, set):
                return list(obj)
            return super().default(obj)
    
    # Create formatter with custom encoder
    custom_encoder_formatter = JsonFormatter(
        fmt='%(asctime)s %(levelname)s %(message)s',
        json_encoder=CustomEncoder
    )
    
    # Test with complex data types
    logger = logging.getLogger('custom_encoder_test')
    logger.setLevel(logging.INFO)
    
    log_stream = io.StringIO()
    handler = logging.StreamHandler(log_stream)
    handler.setFormatter(custom_encoder_formatter)
    
    logger.handlers.clear()
    logger.addHandler(handler)
    
    import datetime
    complex_data = {
        'timestamp': datetime.datetime.now(),
        'tags': {'important', 'user_action', 'database'},
        'nested_object': {'key': 'value', 'number': 42}
    }
    
    logger.info('Complex data test', extra=complex_data)
    
    # Parse custom encoded output
    custom_output = log_stream.getvalue().strip()
    try:
        custom_parsed = json.loads(custom_output)
        print(f"    Custom encoder: ✓ Successfully serialized complex types")
        print(f"      DateTime: {custom_parsed.get('timestamp', 'N/A')[:19]}")
        print(f"      Set as list: {custom_parsed.get('tags', 'N/A')}")
        print(f"      Nested object: {custom_parsed.get('nested_object', 'N/A')}")
        assert custom_parsed['timestamp'] == complex_data['timestamp'].isoformat(), custom_parsed
        assert sorted(custom_parsed['tags']) == sorted(complex_data['tags']), custom_parsed
        assert custom_parsed['nested_object'] == complex_data['nested_object'], custom_parsed
    except json.JSONDecodeError as e:
        # logging.Handler.handleError swallows an encoder TypeError and writes nothing, so a broken encoder lands here.
        raise AssertionError(f"custom-encoded log line is not JSON: {custom_output!r}") from e
    
    # Test formatter options
    print(f"✓ Testing formatter options:")
    
    # Static fields
    static_formatter = JsonFormatter(
        fmt='%(message)s',
        static_fields={'service': 'my-app', 'version': '1.0.0'}
    )
    
    logger_static = logging.getLogger('static_fields_test')
    logger_static.setLevel(logging.INFO)
    
    log_stream_static = io.StringIO()
    handler_static = logging.StreamHandler(log_stream_static)
    handler_static.setFormatter(static_formatter)
    
    logger_static.handlers.clear()
    logger_static.addHandler(handler_static)
    
    logger_static.info('Message with static fields')
    
    static_output = log_stream_static.getvalue().strip()
    try:
        static_parsed = json.loads(static_output)
        print(f"    Static fields: ✓ Added service and version fields")
        print(f"      Service: {static_parsed.get('service')}")
        print(f"      Version: {static_parsed.get('version')}")
        assert static_parsed == {'message': 'Message with static fields', 'service': 'my-app', 'version': '1.0.0'}, static_parsed
    except json.JSONDecodeError as e:
        raise AssertionError(f"static-fields log line is not JSON: {static_output!r}") from e
    
    # Clean up all resources
    for logger_name in ['custom_encoder_test', 'static_fields_test']:
        logger = logging.getLogger(logger_name)
        for handler in logger.handlers[:]:
            logger.removeHandler(handler)
            handler.close()
    
    log_stream.close()
    log_stream_static.close()
    
    print("Example 3 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e).__name__}")
    raise AssertionError(
        f"python_json_logger example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Exception logging and error handling with JSON format

In [ ]:
from pythonjsonlogger.json import JsonFormatter
import logging
import io
import json
import traceback

# Example 4: Exception logging and error handling with JSON format
print("Testing python-json-logger exception and error handling:")

try:
    # Create JSON formatter optimized for error logging
    error_formatter = JsonFormatter(
        fmt='%(asctime)s %(name)s %(levelname)s %(message)s %(pathname)s %(lineno)d'
    )
    
    # Set up error logger
    error_logger = logging.getLogger('error_handler')
    error_logger.setLevel(logging.DEBUG)
    
    log_stream = io.StringIO()
    handler = logging.StreamHandler(log_stream)
    handler.setFormatter(error_formatter)
    
    error_logger.handlers.clear()
    error_logger.addHandler(handler)
    
    print(f"✓ Error logging setup complete")
    
    # Test different error scenarios
    print(f"✓ Testing error scenarios:")
    
    # Scenario 1: Simple error with context
    error_logger.error(
        'Database connection failed',
        extra={
            'database': 'user_db',
            'host': 'localhost',
            'port': 5432,
            'error_code': 'CONN_REFUSED',
            'retry_count': 3
        }
    )
    
    # Scenario 2: Exception with stack trace
    try:
        # Simulate an exception
        x = 1 / 0
    except ZeroDivisionError as e:
        error_logger.exception(
            'Mathematical operation failed',
            extra={
                'operation': 'division',
                'dividend': 1,
                'divisor': 0,
                'function': 'calculate_ratio',
                'user_id': 'user123'
            }
        )
    
    # Scenario 3: Warning with performance data
    error_logger.warning(
        'Slow database query detected',
        extra={
            'query_duration_ms': 5420.3,
            'query_type': 'SELECT',
            'table': 'user_activities',
            'query_id': 'q_456789',
            'threshold_ms': 1000,
            'affected_users': ['user1', 'user2', 'user3']
        }
    )
    
    # Scenario 4: Critical system error
    error_logger.critical(
        'System resource exhausted',
        extra={
            'resource_type': 'memory',
            'current_usage_gb': 15.8,
            'limit_gb': 16.0,
            'usage_percent': 98.75,
            'pid': 12345,
            'service': 'web-server'
        }
    )
    
    print(f"    Logged 4 different error scenarios")
    
    # Parse and analyze error logs
    log_output = log_stream.getvalue()
    log_lines = [line.strip() for line in log_output.split('\n') if line.strip()]
    
    print(f"✓ Error log analysis:")
    print(f"    Generated {len(log_lines)} error log entries")
    
    error_summaries = []
    
    for i, line in enumerate(log_lines):
        try:
            log_entry = json.loads(line)
            
            summary = {
                'level': log_entry.get('levelname'),
                'message': log_entry.get('message'),
                'has_exception': 'exc_info' in log_entry or 'exc_text' in log_entry,
                'extra_field_count': len([k for k in log_entry.keys() 
                                        if k not in ['asctime', 'name', 'levelname', 'message', 'pathname', 'lineno']])
            }
            
            error_summaries.append(summary)
            
            print(f"    Entry {i+1}:")
            print(f"      Level: {summary['level']}")
            print(f"      Message: {summary['message'][:50]}...")
            print(f"      Has exception: {summary['has_exception']}")
            print(f"      Extra fields: {summary['extra_field_count']}")
            
            # Show exception details if present
            if summary['has_exception'] and 'exc_text' in log_entry:
                exc_lines = log_entry['exc_text'].split('\n')
                print(f"      Exception: {exc_lines[-2] if len(exc_lines) > 1 else 'N/A'}")
            
        except json.JSONDecodeError as e:
            print(f"    Entry {i+1}: JSON parse error - {e}")
    
    # Error statistics
    print(f"✓ Error statistics:")
    
    levels = {}
    total_extra_fields = 0
    exception_count = 0
    
    for summary in error_summaries:
        level = summary['level']
        levels[level] = levels.get(level, 0) + 1
        total_extra_fields += summary['extra_field_count']
        if summary['has_exception']:
            exception_count += 1
    
    print(f"    Log levels: {dict(levels)}")
    print(f"    Entries with exceptions: {exception_count}")
    print(f"    Average extra fields per entry: {total_extra_fields / len(error_summaries):.1f}")
    
    # Error handling best practices demonstration
    print(f"✓ Error logging best practices:")
    
    best_practices = [
        "Include contextual information (user_id, session_id, request_id)",
        "Add performance metrics (duration, memory usage, query time)",
        "Structure error data for easy searching and filtering",
        "Include exception stack traces for debugging",
        "Use appropriate log levels (ERROR vs WARNING vs CRITICAL)",
        "Add correlation IDs for distributed tracing",
        "Include environment context (service, host, version)",
        "Sanitize sensitive data before logging"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"    {i}. {practice}")
    
    # Clean up
    error_logger.removeHandler(handler)
    handler.close()
    log_stream.close()
    
    print("Example 4 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e).__name__}")
    # Show traceback for debugging
    traceback.print_exc()
    raise AssertionError(
        f"python_json_logger example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: JSON logging from several threads

In [ ]:
from pythonjsonlogger.json import JsonFormatter
import logging
import io
import json
import threading
import time

# Example 5: JSON logging from several threads
print("Testing python-json-logger logging from several threads:")

try:
    # One JSON formatter configuration shared by every component logger
    production_formatter = JsonFormatter(
        fmt='%(asctime)s %(name)s %(levelname)s %(message)s %(process)d %(thread)d',
        datefmt='%Y-%m-%dT%H:%M:%S.%fZ',
        static_fields={
            'service': 'user-service',
            'version': '2.1.0',
            'environment': 'production',
            'datacenter': 'us-east-1'
        }
    )
    
    print(f"✓ Production formatter configured with static fields")
    
    # Set up multiple loggers for different components
    loggers = {}
    log_streams = {}
    handlers = {}
    
    components = ['auth', 'database', 'api', 'cache', 'metrics']
    
    for component in components:
        logger = logging.getLogger(f'prod_{component}')
        logger.setLevel(logging.INFO)
        
        # Each component gets its own stream for demonstration
        stream = io.StringIO()
        handler = logging.StreamHandler(stream)
        handler.setFormatter(production_formatter)
        
        logger.handlers.clear()
        logger.addHandler(handler)
        
        loggers[component] = logger
        log_streams[component] = stream
        handlers[component] = handler
    
    print(f"✓ Configured {len(components)} production loggers")
    
    # Simulate concurrent logging from multiple threads
    print(f"✓ Testing concurrent logging performance:")
    
    def worker_thread(thread_id, component, count=5):
        """Worker function that generates logs from different components"""
        logger = loggers[component]
        
        for i in range(count):
            logger.info(
                f'Processing request in {component} component',
                extra={
                    'thread_id': thread_id,
                    'request_id': f'{component}_req_{thread_id}_{i}',
                    'operation': f'{component}_operation',
                    'duration_ms': 100 + (i * 10),
                    'status': 'success' if i % 2 == 0 else 'retry',
                    'batch_number': i + 1
                }
            )
            time.sleep(0.001)  # Small delay to simulate processing
    
    # Start multiple threads
    threads = []
    start_time = time.time()
    
    for i, component in enumerate(components):
        thread = threading.Thread(
            target=worker_thread,
            args=(i + 1, component, 3),  # 3 logs per thread
            name=f'Worker-{component}'
        )
        threads.append(thread)
        thread.start()
    
    # Wait for all threads to complete
    for thread in threads:
        thread.join()
    
    end_time = time.time()
    total_duration = end_time - start_time
    
    print(f"    Completed concurrent logging in {total_duration:.3f} seconds")
    
    # Analyze production logs
    print(f"✓ Production log analysis:")
    
    all_logs = []
    total_entries = 0
    
    for component in components:
        stream_output = log_streams[component].getvalue()
        component_lines = [line.strip() for line in stream_output.split('\n') if line.strip()]
        total_entries += len(component_lines)
        
        print(f"    {component}: {len(component_lines)} entries")
        
        # Parse first entry from each component
        if component_lines:
            try:
                sample_entry = json.loads(component_lines[0])
                all_logs.append(sample_entry)
            except json.JSONDecodeError:
                pass
    
    print(f"    Total log entries: {total_entries}")
    print(f"    Average entries per component: {total_entries / len(components):.1f}")
    
    # Verify production log structure
    if all_logs:
        sample_log = all_logs[0]
        print(f"✓ Production log structure verification:")
        
        required_fields = ['asctime', 'name', 'levelname', 'message', 'process', 'thread']
        static_fields = ['service', 'version', 'environment', 'datacenter']
        extra_fields = ['request_id', 'operation', 'duration_ms', 'status']
        
        for field_category, fields in [
            ('Required', required_fields),
            ('Static', static_fields),
            ('Extra', extra_fields)
        ]:
            present_fields = [f for f in fields if f in sample_log]
            print(f"    {field_category} fields: {len(present_fields)}/{len(fields)} present")
            if len(present_fields) < len(fields):
                missing = [f for f in fields if f not in sample_log]
                print(f"      Missing: {missing}")
    
    # Performance metrics
    print(f"✓ Performance metrics:")
    
    logs_per_second = total_entries / total_duration if total_duration > 0 else 0
    print(f"    Logs per second: {logs_per_second:.1f}")
    print(f"    Average log size: {sum(len(json.dumps(log)) for log in all_logs) / len(all_logs):.1f} chars")
    
    # Production logging best practices
    print(f"✓ Production logging best practices demonstrated:")
    
    practices = [
        "✓ Structured JSON format for log aggregation tools",
        "✓ Static fields for service identification",
        "✓ Thread-safe logging for concurrent applications", 
        "✓ Process and thread IDs for debugging",
        "✓ Consistent timestamp format (ISO 8601)",
        "✓ Request correlation IDs for tracing",
        "✓ Performance metrics (duration, status)",
        "✓ Component-specific loggers for organization"
    ]
    
    for practice in practices:
        print(f"    {practice}")
    
    # Recommended production setup
    print(f"✓ Recommended production setup:")
    
    recommendations = [
        "Use centralized logging (ELK stack, Splunk, etc.)",
        "Configure log rotation to manage disk space",
        "Set appropriate log levels per environment",
        "Use structured logging for all application components",
        "Include correlation IDs for distributed tracing",
        "Monitor log volume and performance impact",
        "Implement log sampling for high-volume services",
        "Use async handlers for high-performance applications"
    ]
    
    for i, rec in enumerate(recommendations, 1):
        print(f"    {i}. {rec}")
    
    # Clean up all resources
    for component in components:
        logger = loggers[component]
        handler = handlers[component]
        stream = log_streams[component]
        
        logger.removeHandler(handler)
        handler.close()
        stream.close()
    
    print("Example 5 completed successfully")
    
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    raise